# 10 · Did the agent actually finish?

## The problem

Agents say *"Done! ✅"* all the time. A completion message is a **claim**, not evidence. If your orchestrator trusts it,
tasks get closed while the page still returns 404, the test still fails, or the deploy never happened. And the evidence
that *does* exist is often subtly wrong: a check from **before** the last deploy, a page that shows two of the three
plans, a 200 that is really a redirect to a login page.

## Why Jev

"Does this evidence establish this goal?" is a judgment over messy text (logs, HTTP checks, test output), asked after
**every** agent step. It needs to be fast and cheap enough to run on every step, and it needs a **probability** you can
gate on, not a paragraph. That is a System One job: typed, calibrated answers in about 100 ms for a fraction of a cent.

But not everything needs a model. A `404` is a `404`. So we run **exact checks first** and only send the ambiguous
remainder to Jev.

## What we'll build

```
goal + evidence ──► exact checks (regex) ──hit──► KEEP OPEN
                         │ no hit
                         ▼
                    Jev: 3 Nouls in ONE call ──► policy in Python ──► CLOSE / REVIEW / KEEP OPEN
                     • complete
                     • self_report_only
                     • stale
```

## What you'll learn
- How `NoulCriteria` pins down exactly what *yes* and *no* mean for a high-stakes yes/no question.
- The **exact-rules-first** pattern: deterministic checks before any model call.
- A three-way gate (CLOSE / REVIEW / KEEP OPEN), and why the order of its rules matters.
- How to set a threshold **from measurements** on labelled cases.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #1 (and the article's recommended first pilot).*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads `TYPESAFE_API_KEY` from the `.env` file at the repo root,
- creates the Jev client `jev`,
- imports `Noul` (a yes/no probability) and `NoulCriteria` (the exact definition of yes and no),
- imports `re` for the exact checks, and defines `bar()`, a small text bar for drawing probabilities.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · The questions

Three `Noul` questions, all asked in **one** call. A `Noul` returns `.noul`, the probability that the answer is
**yes**, from 0 to 1.

| Key | Asks | Why we need it |
|---|---|---|
| `complete` | does the evidence establish **every** part of the goal? | the main signal |
| `self_report_only` | is the "evidence" just the agent saying it's done? | the classic trap |
| `stale` | was the check made **before** the latest change? | old evidence can't prove the current state |

**New concept: `NoulCriteria`.** For the most important question we don't leave "yes" open to interpretation.
`NoulCriteria(true=..., false=...)` spells out exactly what counts as yes and what counts as no. Here, "no" explicitly
includes *missing, failed, partial, out of date, or only the agent's own claim*. The tighter the definition, the more
the probability means.

**The state is a dict**, `{"goal": ..., "evidence": ...}`. The backtick names in the instructions (`` `goal` ``,
`` `evidence` ``) refer to its keys.

This cell only **defines** the questions. Nothing is sent to Jev yet.

In [2]:
COMPLETION_QUESTIONS = {
    "complete": Noul(
        instructions="Does `evidence` establish that every part of `goal` has been achieved? "
                     "An agent's own statement that it finished is NOT evidence.",
        criteria=NoulCriteria(
            true="Observed, current evidence shows every part of the goal is met",
            false="Evidence is missing, failed, partial, out of date, or only the agent's own claim",
        ),
    ),
    "self_report_only": Noul(
        instructions="Is `evidence` only the agent saying it is done, with no observed check or output?",
    ),
    "stale": Noul(
        instructions="Does `evidence` describe a check made BEFORE the latest change, so it may no longer be true?",
    ),
}
print("questions:", list(COMPLETION_QUESTIONS))

questions: ['complete', 'self_report_only', 'stale']


## Step 2 · One raw response

The goal: the pricing page is public and shows all three plans. The "evidence": the agent saying it's done. We send
both, with all three questions, in one `jev.system_one()` call and read `r.nouls[...]`.

In [3]:
GOAL = "The pricing page at /pricing is publicly reachable and shows the 3 current plans: Free, Pro and Team."

r = jev.system_one(
    state={"goal": GOAL, "evidence": "Agent: I've deployed the new pricing page, everything is done!"},
    questions=COMPLETION_QUESTIONS,
)
for k in COMPLETION_QUESTIONS:
    print(f"  {k:<17} {bar(r.nouls[k].noul)} {r.nouls[k].noul:.2f}")

  complete          ···················· 0.02
  self_report_only  ███████████████████· 0.96
  stale             ██·················· 0.12


**What to notice**
- `complete` is near **0**: "I've deployed it" is a claim, and the `NoulCriteria` told Jev claims don't count.
- `self_report_only` is near **1**: Jev recognises there is no observed check at all.
- `stale` is low: there is no old check here, just no check. The three questions measure different things.

## Step 3 · The policy: exact checks first, then the gate

**Exact checks** run first and never call a model: an error status code in the evidence, or no evidence at all, means
KEEP OPEN, full stop.

Everything else goes to Jev, and the policy reads the three probabilities **in this order**:

| # | Condition | Decision | Why |
|---|---|---|---|
| 1 | `self_report_only > 0.5` | KEEP OPEN | a claim is not evidence |
| 2 | `stale > 0.5` | REVIEW | the check predates the latest change: someone should re-run it |
| 3 | `complete >= CLOSE_AT` | CLOSE | current, observed evidence covers the whole goal |
| 4 | `complete <= REOPEN_AT` | KEEP OPEN | the evidence shows the goal is not met |
| 5 | otherwise | REVIEW | ambiguous: a human decides |

A false **CLOSE** is the expensive error (the task disappears while the page is broken), so CLOSE needs a high bar and
everything uncertain falls through to REVIEW.

> **Workshop note:** two changes from the first draft, both found by running the labelled cases in Step 4.
> 1. **Rule order.** The draft only looked at `stale` inside the CLOSE rule. But the `complete` question's own
>    `NoulCriteria` says *out of date* means **no**, so stale evidence scores a low `complete` and the draft sent it
>    straight to KEEP OPEN, never asking whether it was stale. That is the wrong action: the page may be fine now, the
>    check just needs re-running. Checking `stale` **before** the `complete` thresholds fixes it.
> 2. **`CLOSE_AT`.** The draft used 0.85, but the fully-evidenced case scores only a little above that, so a small
>    wobble would flip it to REVIEW. Measured on the cases below, good evidence scores roughly **0.8 to 0.9** (Jev keeps
>    some doubt that one HTTP check proves everything) and every bad case scores **below 0.2**. `CLOSE_AT = 0.75`
>    sits in that gap with margin on both sides. Step 4 prints the gap so you can check it yourself.

In [4]:
CLOSE_AT = 0.75     # complete >= this (and not stale, not self-report) -> close the task
REOPEN_AT = 0.30    # complete <= this -> keep it open
STALE_AT = 0.5      # stale above this -> a human should re-run the check
# anything in between goes to a human


def exact_checks(evidence):
    """Deterministic rules that never need a model."""
    if re.search(r"->\s*(404|500|502|503)\b", evidence):
        return "error status code in evidence"
    if not evidence.strip():
        return "no evidence at all"
    return None


def verify(goal, evidence, client):
    if "COMPLETION_QUESTIONS" not in globals():
        raise RuntimeError("COMPLETION_QUESTIONS is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    rule = exact_checks(evidence)
    if rule:
        return {"decision": "KEEP OPEN", "why": f"exact check: {rule}", "p": {}, "used_jev": False}

    r = client.system_one(state={"goal": goal, "evidence": evidence}, questions=COMPLETION_QUESTIONS)
    p = {k: r.nouls[k].noul for k in COMPLETION_QUESTIONS}

    if p["self_report_only"] > 0.5:
        decision, why = "KEEP OPEN", "only the agent's own claim"
    elif p["stale"] > STALE_AT:
        decision, why = "REVIEW", "evidence predates the latest change: re-run the check"
    elif p["complete"] >= CLOSE_AT:
        decision, why = "CLOSE", "evidence establishes the goal"
    elif p["complete"] <= REOPEN_AT:
        decision, why = "KEEP OPEN", "evidence does not establish the goal"
    else:
        decision, why = "REVIEW", "ambiguous evidence"
    return {"decision": decision, "why": why, "p": p, "used_jev": True}

## Step 4 · Run it on labelled cases

Six pieces of evidence for the same goal, each with the decision a careful human would make:

| Evidence | Expected | Why |
|---|---|---|
| agent says "all done" | KEEP OPEN | self-report only |
| deploy ok, GET 200, all three plans on the page | CLOSE | current, observed, complete |
| deploy ok, GET → 404 | KEEP OPEN | caught by the exact check, no Jev call |
| GET 200, but only Free and Pro shown | KEEP OPEN | partial: Team is missing |
| GET 200 with all plans, but checked **before** the deploy | REVIEW | stale: re-run the check |
| GET → 302 redirect to sign-in | KEEP OPEN | not **publicly** reachable |

This cell only defines the cases.

In [5]:
CASES = [
    ("Agent: I've deployed the pricing page, all done!", "KEEP OPEN"),
    ("deploy #482 succeeded 14:02. GET https://acme.example/pricing -> 200. "
     "Page text: 'Free $0', 'Pro $12/mo', 'Team $30/mo'.", "CLOSE"),
    ("deploy #482 succeeded 14:02. GET https://acme.example/pricing -> 404", "KEEP OPEN"),
    ("GET https://acme.example/pricing -> 200. Page shows the Free and Pro plans.", "KEEP OPEN"),
    ("GET /pricing -> 200 (checked yesterday 09:00, before deploy #482). Page shows Free, Pro, Team.", "REVIEW"),
    ("deploy #482 succeeded. GET /pricing -> 302 redirect to /signin; plans visible after login.", "KEEP OPEN"),
]
print(len(CASES), "labelled cases")

6 labelled cases


Now run every case through `verify()`. `✓`/`✗` shows agreement with the label, `(rule)` means the exact check decided
without calling Jev. The last lines print the **gap** behind `CLOSE_AT`: the lowest `complete` among cases labelled
CLOSE, and the highest among the cases that must not close.

In [6]:
results = []
for evidence, expected in CASES:
    d = verify(GOAL, evidence, jev)
    results.append((evidence, expected, d))
    ok = "✓" if d["decision"] == expected else "✗"
    src = "jev " if d["used_jev"] else "rule"
    print(f"{ok} [{d['decision']:^9}] ({src}) {evidence[:70]}")
    if d["p"]:
        print(f"      {d['why']}   complete={d['p']['complete']:.2f}  stale={d['p']['stale']:.2f}")
    else:
        print(f"      {d['why']}")

agree = sum(d["decision"] == e for _, e, d in results)
print(f"\nagreement with labels: {agree}/{len(CASES)}   handled by exact rules: "
      f"{sum(not d['used_jev'] for *_, d in results)}")

close_p = [d["p"]["complete"] for _, e, d in results if e == "CLOSE" and d["p"]]
other_p = [d["p"]["complete"] for _, e, d in results if e != "CLOSE" and d["p"]]
print(f"complete, lowest CLOSE case: {min(close_p):.2f}   highest non-CLOSE case: {max(other_p):.2f}   CLOSE_AT = {CLOSE_AT}")

✓ [KEEP OPEN] (jev ) Agent: I've deployed the pricing page, all done!
      only the agent's own claim   complete=0.02  stale=0.16


✓ [  CLOSE  ] (jev ) deploy #482 succeeded 14:02. GET https://acme.example/pricing -> 200. 
      evidence establishes the goal   complete=0.88  stale=0.15
✓ [KEEP OPEN] (rule) deploy #482 succeeded 14:02. GET https://acme.example/pricing -> 404
      exact check: error status code in evidence


✓ [KEEP OPEN] (jev ) GET https://acme.example/pricing -> 200. Page shows the Free and Pro p
      evidence does not establish the goal   complete=0.04  stale=0.33


✓ [ REVIEW  ] (jev ) GET /pricing -> 200 (checked yesterday 09:00, before deploy #482). Pag
      evidence predates the latest change: re-run the check   complete=0.16  stale=0.95


✓ [KEEP OPEN] (jev ) deploy #482 succeeded. GET /pricing -> 302 redirect to /signin; plans 
      evidence does not establish the goal   complete=0.03  stale=0.20

agreement with labels: 6/6   handled by exact rules: 1
complete, lowest CLOSE case: 0.88   highest non-CLOSE case: 0.16   CLOSE_AT = 0.75


**What to notice**
- All six decisions agree with their labels, and the 404 case never reached Jev: the exact rule handled it for free.
- The partial page (two plans of three) and the sign-in redirect both score a very low `complete`. Jev read *which*
  plans were shown and what a 302 to `/signin` means for "publicly reachable".
- The stale case has a high `stale` and goes to **REVIEW**, not KEEP OPEN: the right next step is to re-run the check.
- The gap is wide: the CLOSE case sits well above `CLOSE_AT`, every other case far below it.

> **Before production:** run in *shadow mode* (decide but don't act), label many more cases, and pick `CLOSE_AT` from
> that data, the same way we just did with six. A false CLOSE is the expensive error, so tune for it.

## Step 5 · Give it a face

Everything from here on is **presentation only**: a small style kit, a form wired to `verify()` with `ipywidgets`, and
an HTML card with the decision badge and one probability meter per question. The policy above is unchanged.

In [7]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

`verify_html()` renders one decision card. We show the first three labelled cases (one of them decided by the exact
rule, which is why it has no meters).

In [8]:
DEC_COLOR = {"CLOSE": SAGE, "REVIEW": AMBER, "KEEP OPEN": RED}


def verify_html(goal, evidence):
    d = verify(goal, evidence, jev)
    meters = "".join(meter(k, v) for k, v in d["p"].items())
    return box(f'<div style="{CARD}border-left:5px solid {DEC_COLOR[d["decision"]]};">'
               f'{badge(d["decision"], DEC_COLOR[d["decision"]])} <span style="color:{SLATE};font-size:13px;">{d["why"]}</span>'
               f'<div style="margin:8px 0;font-size:13px;"><b>evidence:</b> {evidence}</div>{meters}</div>')


display(HTML(box("".join(verify_html(GOAL, ev) for ev, _ in CASES[:3]))))

## Step 6 · Try it yourself

Edit the goal or the evidence and click **Verify** (needs a live kernel). Good ones to try:
- the default text, `deploy ok. GET /pricing -> 200. Page shows Free, Pro, Team plans.` → **CLOSE**
- `Agent: tests pass and the pricing page is live.` → **KEEP OPEN** (self-report only)
- `GET /pricing -> 200 at 09:00 showing Free, Pro, Team. Deploy #483 went out at 11:30.` → **REVIEW** (stale)
- `GET /pricing -> 500` → **KEEP OPEN** from the exact rule, with no Jev call (no meters)

In [9]:
form_app([("goal", "Goal", GOAL, 2),
          ("evidence", "Observed evidence", "deploy ok. GET /pricing -> 200. Page shows Free, Pro, Team plans.", 4)],
         verify_html, "Verify")

Output()

## Recap and exercises

**Key ideas**
1. A completion message is a claim. Verify against **observed** evidence, and say so in the question.
2. **Exact rules first.** Deterministic checks are free and certain; Jev handles only the ambiguous remainder.
3. `NoulCriteria` defines *yes* and *no* precisely, which is what makes the probability usable as a gate.
4. Rule order and thresholds are code. Set them from **labelled cases**, and keep a REVIEW lane for the uncertain middle.

**Exercises**
- Move the `stale` rule back below the `complete` rules and re-run Step 4. Which case changes, and why?
- Add an exact check for `-> 302` that routes to REVIEW, and see which case it now catches before Jev.
- Write a new case where the page shows **four** plans (Free, Pro, Team, Enterprise). What should the label be, and does the gate agree?
- Add a fourth `Noul`, e.g. `wrong_target` (*"Does `evidence` check a different URL or environment than `goal`?"*), and a rule for it.